In [ ]:
import numpy as np
import random as rd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import math
from scipy.stats import poisson
from itertools import product
from copy import deepcopy

# Tabular Solution Methods 

## Multi-armed Bandits

### 10-armed Testbed

In [ ]:
rng = np.random.default_rng()

In [ ]:
NtestsAB = 2000
kAB = 10
qvaluesAB = rng.standard_normal((NtestsAB, kAB))
qvaluesNSAB = np.ones((NtestsAB, kAB)) * qvaluesAB[0, :]

### Action-value Methods

In [ ]:
def sampleAverage(maxIter, epsilon=0, NS=False, alpha=0, qInit=0, beta=False, UCB=0):
    Q = qInit * np.ones((NtestsAB, kAB))
    N = np.zeros((NtestsAB, kAB), dtype=int)
    AvgReward = np.zeros(maxIter)
    AvgOptimal = np.zeros(maxIter)
    nExplore = np.zeros(maxIter)
    qvaluesCurr = qvaluesNSAB if NS else qvaluesAB
    qstarsCurr = np.max(qvaluesCurr, axis=1)
    if beta:
        oHat = 1 - (1-alpha)**np.arange(maxIter+1)

    for t in range(maxIter):
        if UCB:
            offset = UCB * np.sqrt(np.log(t+1)/N)
            offset[N == 0] = np.inf
        At = np.argmax(Q + (0 if not UCB else offset), axis=1)
        explore = rng.random(NtestsAB) <= epsilon
        nExplore[t] = explore.sum()
        randomAt = rng.integers(kAB-1, size=NtestsAB)
        randomAt += randomAt >= At
        At[explore] = randomAt[explore]
        qvaluesAt = qvaluesCurr[np.arange(NtestsAB), At]
        AvgOptimal[t] = np.mean(qvaluesCurr[np.arange(NtestsAB), At] == qstarsCurr)
        rewards = rng.normal(loc=qvaluesAt, scale=1)
        AvgReward[t] = np.mean(rewards)
        N[np.arange(NtestsAB),At] += 1
        increment = 1 / N[np.arange(NtestsAB), At]
        if alpha:
            increment = alpha
            if beta:
                increment = alpha / oHat[N[np.arange(NtestsAB), At]]
        Q[np.arange(NtestsAB),At] = Q[np.arange(NtestsAB), At] + increment * (rewards - Q[np.arange(NtestsAB), At])
        if NS:
            qvaluesCurr += rng.normal(loc=np.zeros((NtestsAB, kAB)), scale=0.01)
            qstarsCurr = np.max(qvaluesCurr, axis=1)

    return Q, AvgReward, AvgOptimal, nExplore
            

In [ ]:
def gradientBandit(maxIter, baseline=True, NS=False, alpha=0.1):
    H = np.zeros((NtestsAB, kAB))
    R = np.zeros(NtestsAB)
    AvgReward = np.zeros(maxIter)
    AvgOptimal = np.zeros(maxIter)
    qvaluesCurr = qvaluesNSAB if NS else qvaluesGB
    qstarsCurr = np.max(qvaluesCurr, axis=1)

    for t in range(maxIter):
        pi = np.exp(H)
        pi /= pi.sum(axis=1, keepdims=True)
        u = rng.random(NtestsAB)
        At = (np.cumsum(pi, axis=1) < u[:, None]).sum(axis=1)
        qvaluesAt = qvaluesCurr[np.arange(NtestsAB), At]
        AvgOptimal[t] = np.mean(qvaluesCurr[np.arange(NtestsAB), At] == qstarsCurr)
        rewards = rng.normal(loc=qvaluesAt, scale=1)
        AvgReward[t] = np.mean(rewards)
        bl = np.zeros(NtestsAB) if not baseline else (rewards if t == 0 else R / t)
        H[np.arange(NtestsAB), At] = H[np.arange(NtestsAB), At] + alpha * (rewards - bl) * (1 - pi[np.arange(NtestsAB), At])
        H[np.arange(kAB) != At[:, None]] -= (alpha * (rewards[:, None] - bl[:, None]) * pi)[np.arange(kAB) != At[:, None]]
        R += rewards
        if NS:
            qvaluesCurr += rng.normal(loc=np.zeros((NtestsAB, kAB)), scale=0.01)
            qstarsCurr = np.max(qvaluesCurr, axis=1)

    return H, AvgReward, AvgOptimal


In [ ]:
nSampleAB = 1000
Qsample, AvgRSample, AvgOptSample, nExploreSample = sampleAverage(nSampleAB)
Qsample01, AvgRSample01, AvgOptSample01, nExploreSample01 = sampleAverage(nSampleAB, epsilon=0.1)
Qsample001, AvgRSample001, AvgOptSample001, nExploreSample001 = sampleAverage(nSampleAB, epsilon=0.01)
fig = go.Figure()
fig.add_trace(go.Scatter(x=np.arange(nSampleAB), y=AvgRSample, name="greedy"))
fig.add_trace(go.Scatter(x=np.arange(nSampleAB), y=AvgRSample01, name="0.1-greedy"))
fig.add_trace(go.Scatter(x=np.arange(nSampleAB), y=AvgRSample001, name="0.01-greedy"))
fig.show()
fig = go.Figure()
fig.add_trace(go.Scatter(x=np.arange(nSampleAB), y=AvgOptSample, name="greedy"))
fig.add_trace(go.Scatter(x=np.arange(nSampleAB), y=AvgOptSample01, name="0.1-greedy"))
fig.add_trace(go.Scatter(x=np.arange(nSampleAB), y=AvgOptSample001, name="0.01-greedy"))
fig.show()

In [ ]:
nSampleAB = 1000
Qsample01NS, AvgRSample01NS, AvgOptSample01NS, nExploreSample01NS = sampleAverage(nSampleAB, epsilon=0.1, NS=True)
Qalpha01NS, AvgRalpha01NS, AvgOptalpha01NS, nExploreSample01NS = sampleAverage(nSampleAB, epsilon=0.1, NS=True, alpha=0.1)
fig = go.Figure()
fig.add_trace(go.Scatter(x=np.arange(nSampleAB), y=AvgRSample01NS, name="0.1-greedy-avgMean"))
fig.add_trace(go.Scatter(x=np.arange(nSampleAB), y=AvgRalpha01NS, name="0.1-greedy-alpha"))
fig.show()
fig = go.Figure()
fig.add_trace(go.Scatter(x=np.arange(nSampleAB), y=AvgOptSample01NS, name="0.1-greedy-avgMean"))
fig.add_trace(go.Scatter(x=np.arange(nSampleAB), y=AvgOptalpha01NS, name="0.1-greedy-alpha"))
fig.show()

In [ ]:
nSampleAB = 1000
Qalphaqinit, AvgRalphaqinit, AvgOptalphaqinit, nExploreSampleqinit = sampleAverage(nSampleAB, alpha=0.1, qInit=5)
Qalpha01, AvgRalpha01, AvgOptalpha01, nExplorealpha01 = sampleAverage(nSampleAB, epsilon=0.1, alpha=0.1)

fig = go.Figure()
fig.add_trace(go.Scatter(x=np.arange(nSampleAB), y=AvgRalphaqinit, name="greedy-alpha-qinit"))
fig.add_trace(go.Scatter(x=np.arange(nSampleAB), y=AvgRalpha01, name="0.1-greedy-alpha"))
fig.show()

fig = go.Figure()
fig.add_trace(go.Scatter(x=np.arange(nSampleAB), y=AvgOptalphaqinit, name="greedy-alpha-qinit"))
fig.add_trace(go.Scatter(x=np.arange(nSampleAB), y=AvgOptalpha01, name="0.1-greedy-alpha"))
fig.show()


In [ ]:
nSampleAB = 1000
Qalphaqinit, AvgRalphaqinit, AvgOptalphaqinit, nExploreSampleqinit = sampleAverage(nSampleAB, alpha=0.1, qInit=5)
QalphaqinitO5, AvgRalphaqinitO5, AvgOptalphaqinitO5, nExploreSampleqinitO5 = sampleAverage(nSampleAB, alpha=0.1, qInit=5, beta=True)
QalphaqinitO10, AvgRalphaqinitO10, AvgOptalphaqinitO10, nExploreSampleqinitO10 = sampleAverage(nSampleAB, alpha=0.1, qInit=10, beta=True)

fig = go.Figure()
fig.add_trace(go.Scatter(x=np.arange(nSampleAB), y=AvgRalphaqinit, name="greedy-alpha-qinit-5"))
fig.add_trace(go.Scatter(x=np.arange(nSampleAB), y=AvgRalphaqinitO5, name="greedy-beta-qinit-5"))
fig.add_trace(go.Scatter(x=np.arange(nSampleAB), y=AvgRalphaqinitO10, name="greedy-beta-qinit-10"))
fig.show()

fig = go.Figure()
fig.add_trace(go.Scatter(x=np.arange(nSampleAB), y=AvgOptalphaqinit, name="greedy-alpha-qinit-5"))
fig.add_trace(go.Scatter(x=np.arange(nSampleAB), y=AvgOptalphaqinitO5, name="greedy-beta-qinit-5"))
fig.add_trace(go.Scatter(x=np.arange(nSampleAB), y=AvgOptalphaqinitO10, name="greedy-beta-qinit-10"))
fig.show()

In [ ]:
nSampleAB = 1000
QalphaUCB2, AvgRalphaUCB2, AvgOptalphaUCB2, nExploreSampleUCB2 = sampleAverage(nSampleAB, UCB=2)
QalphaUCB1, AvgRalphaUCB1, AvgOptalphaUCB1, nExploreSampleUCB1 = sampleAverage(nSampleAB, UCB=1)
Qalpha01, AvgRalpha01, AvgOptalpha01, nExplorealpha01 = sampleAverage(nSampleAB, epsilon=0.1)

fig = go.Figure()
fig.add_trace(go.Scatter(x=np.arange(nSampleAB), y=AvgRalphaUCB2, name="greedy-Avg-UCB-2"))
fig.add_trace(go.Scatter(x=np.arange(nSampleAB), y=AvgRalphaUCB1, name="greedy-Avg-UCB-1"))
fig.add_trace(go.Scatter(x=np.arange(nSampleAB), y=AvgRalpha01, name="0.1-greedy-Avg"))
fig.show()

fig = go.Figure()
fig.add_trace(go.Scatter(x=np.arange(nSampleAB), y=AvgOptalphaUCB2, name="greedy-Avg-UCB-2"))
fig.add_trace(go.Scatter(x=np.arange(nSampleAB), y=AvgOptalphaUCB1, name="greedy-Avg-UCB-1"))
fig.add_trace(go.Scatter(x=np.arange(nSampleAB), y=AvgOptalpha01, name="0.1-greedy-Avg"))
fig.show()

In [ ]:
qvaluesGB = rng.normal(loc=4, scale=1, size=((NtestsAB, kAB)))
nSampleAB = 1000
HGBalpha01bl0, AvgRGBalpha01bl0, AvgOptGBalpha01bl0 = gradientBandit(nSampleAB, baseline=False, alpha=0.1)
HGBalpha04bl0, AvgRGBalpha04bl0, AvgOptGBalpha04bl0 = gradientBandit(nSampleAB, baseline=False, alpha=0.4)
HGBalpha01bl, AvgRGBalpha01bl, AvgOptGBalpha01bl = gradientBandit(nSampleAB, baseline=True, alpha=0.1)
HGBalpha04bl, AvgRGBalpha04bl, AvgOptGBalpha04bl = gradientBandit(nSampleAB, baseline=True, alpha=0.4)

fig = go.Figure()
fig.add_trace(go.Scatter(x=np.arange(nSampleAB), y=AvgRGBalpha01bl0, name="GB-alpha-0.1-BL-0"))
fig.add_trace(go.Scatter(x=np.arange(nSampleAB), y=AvgRGBalpha04bl0, name="GB-alpha-0.4-BL-0"))
fig.add_trace(go.Scatter(x=np.arange(nSampleAB), y=AvgRGBalpha01bl, name="GB-alpha-0.1-BL"))
fig.add_trace(go.Scatter(x=np.arange(nSampleAB), y=AvgRGBalpha04bl, name="GB-alpha-0.4-BL"))
fig.show()

fig = go.Figure()
fig.add_trace(go.Scatter(x=np.arange(nSampleAB), y=AvgOptGBalpha01bl0, name="GB-alpha-0.1-BL-0"))
fig.add_trace(go.Scatter(x=np.arange(nSampleAB), y=AvgOptGBalpha04bl0, name="GB-alpha-0.4-BL-0"))
fig.add_trace(go.Scatter(x=np.arange(nSampleAB), y=AvgOptGBalpha01bl, name="GB-alpha-0.1-BL"))
fig.add_trace(go.Scatter(x=np.arange(nSampleAB), y=AvgOptGBalpha04bl, name="GB-alpha-0.4-BL"))
fig.show()

### Parameter study

In [ ]:
# nSampleAB = 1000
# # Epsilon greedy
# epsilons = 0.4 * (2.0 ** np.arange(-7, 0))
# epsilonGreedy = [sampleAverage(nSampleAB, epsilon=epsilon) for epsilon in epsilons]

# # Gradient Bandit
# qvaluesGB = rng.normal(loc=0, scale=1, size=((NtestsAB, kAB)))
# alphasGB = 0.8 * (2.0 ** np.arange(-5, 3))
# GB = [gradientBandit(nSampleAB, baseline=True, alpha=alpha) for alpha in alphasGB]

# # UCB
# UCBcs = 0.5 * (2.0 ** np.arange(-4, 4))
# UCBs = [sampleAverage(nSampleAB, UCB=c) for c in UCBcs]

# # Greedy optimistic
# qinits = 0.5 * (2.0 ** np.arange(-2, 4))
# optimisticGreedy = [sampleAverage(nSampleAB, alpha=0.1, qinit=qinit) for qinit in qinits]

In [ ]:
# fig = go.Figure()
# fig.add_trace(go.Scatter(x=np.log2(epsilons/0.4), y=[epsilonGreedy[i][1].mean() for i in range(len(epsilons))], name="epsilon-greedy"))
# fig.add_trace(go.Scatter(x=np.log2(alphasGB/0.8), y=[GB[i][1].mean() for i in range(len(alphasGB))], name="gradient bandit"))
# fig.add_trace(go.Scatter(x=np.log2(UCBcs/0.5), y=[UCBs[i][1].mean() for i in range(len(UCBcs))], name="UCB"))
# fig.add_trace(go.Scatter(x=np.log2(qinits/0.5), y=[optimisticGreedy[i][1].mean() for i in range(len(qinits))], name="greedy optimistic alpha=0.1"))
# fig.show()

In [ ]:
# nSampleNSAB = 200000
# # Epsilon greedy
# epsilons = 0.4 * (2.0 ** np.arange(-7, 0))
# epsilonGreedy = [sampleAverage(nSampleNSAB, epsilon=epsilon, NS=True) for epsilon in epsilons]

In [ ]:
# # Epsilon greedy constant alpha
# epsilonGreedyAlpha = [sampleAverage(nSampleNSAB, epsilon=epsilon, NS=True, alpha=0.1) for epsilon in epsilons]

In [ ]:
# # Gradient Bandit
# qvaluesGB = rng.normal(loc=0, scale=1, size=((Ntests, k)))
# alphasGB = 0.8 * (2.0 ** np.arange(-5, 3))
# GB = [gradientBandit(nSampleNSAB, baseline=True, NS=True, alpha=alpha) for alpha in alphasGB]

In [ ]:
# # UCB
# UCBcs = 0.5 * (2.0 ** np.arange(-4, 4))
# UCBs = [sampleAverage(nSampleNSAB, NS=True, UCB=c) for c in UCBcs]

In [ ]:
# # Greedy optimistic
# qinits = 0.5 * (2.0 ** np.arange(-2, 4))
# optimisticGreedy = [sampleAverage(nSampleNSAB, NS=True, alpha=0.1, qinit=qinit) for qinit in qinits]

In [ ]:
# fig = go.Figure()
# fig.add_trace(go.Scatter(x=np.log2(epsilons/0.4), y=[epsilonGreedy[i][1][100000:].mean() for i in range(len(epsilons))], name="epsilon-greedy"))
# fig.add_trace(go.Scatter(x=np.log2(epsilons/0.4), y=[epsilonGreedyAlpha[i][1][100000:].mean() for i in range(len(epsilons))], name="epsilon-greedy alpha=0.1"))
# fig.add_trace(go.Scatter(x=np.log2(alphasGB/0.8), y=[GB[i][1][100000:].mean() for i in range(len(alphasGB))], name="gradient bandit"))
# fig.add_trace(go.Scatter(x=np.log2(UCBcs/0.5), y=[UCBs[i][1][100000:].mean() for i in range(len(UCBcs))], name="UCB"))
# fig.add_trace(go.Scatter(x=np.log2(qinits/0.5), y=[optimisticGreedy[i][1][100000:].mean() for i in range(len(qinits))], name="greedy optimistic alpha=0.1"))
# fig.show()

## Finite Markov Decision Processes

### Policy iteration

In [ ]:
# Policy Evaluation
def Ld(model, s, a, gamma, V):
    nStates, actionMapping, rewardMapping, dynamics, terminals = model
    return rewardMapping[s, a] + gamma * sum(dynamics[s, a, sPrime] * V[sPrime] for sPrime in range(nStates))

def policyEval(model, policy, V, theta, gamma):
    nStates, actionMapping, rewardMapping, dynamics, terminals = model
    converged = False
    while not converged:
        delta = 0
        for s in range(nStates):
            v = V[s]
            V[s] = Ld(model, s, policy[s], gamma, V)
            delta = max(delta, abs(v - V[s]))
        converged = delta < theta


In [ ]:
# Policy Improvement
def policyImprov(model, policy, V, gamma):
    policyStable = True
    nStates, actionMapping, rewardMapping, dynamics, terminals = model
    for s in range(nStates):
        oldPolicy = policy[s]
        # Greedy action improv
        maxi = Ld(model, s, oldPolicy, gamma, V)
        maxiAction = policy[s]
        for a in actionMapping[s]:
            qValAction = Ld(model, s, a, gamma, V)
            if qValAction > maxi:
                maxi = qValAction
                maxiAction = a

        policy[s] = maxiAction
        
        if policy[s] != oldPolicy:
            policyStable = False
    return policyStable
        

In [ ]:
def policyIter(model, theta, gamma, policyInit, vInit):
    # Initialization
    nStates, actionMapping, rewardMapping, dynamics, terminals = model
    V = vInit.copy()
    V[terminals] = np.zeros(len(terminals))
    policy = policyInit.copy()
    policyStable = False
    iter = 0
    while not policyStable:
        # Policy Evaluation
        print(f"iteration {iter}")
        policyEval(model, policy, V, theta, gamma)

        # Policy Improv
        policyStable = policyImprov(model, policy, V, gamma)

        iter += 1

    return V, policy

#### Example 4.2 : Jack's Car Rental

In [ ]:
# Parameters
maxCars = 20
maxMoves = 5
nStatesJCR = (maxCars + 1) ** 2
freeRide = False
maxParkingSpace = 20
parkingFee = 4
movingFee = 2
lambdaRentLoc1, lambdaRentLoc2, lambdaRetLoc1, lambdaRetLoc2 = 3, 4, 3, 2

We start by defining the probabilities $P^{inter}_1, P^{inter}_2$ to go from one intermediary state $s_{inter}$ to a final state $s'$ at locations 1 and 2 respectively. Intermediary states incode the number of cars present at locations 1 and 2 after all overnight movements are completed. Final states incode the number of cars present at locations 1 and 2 at the end of the work day, so that $P^{inter}_1, P^{inter}_2$ are the probabilities to start a day at $s^{inter}$ and end it at $s'$.

Let $n^{inter}_1, n^{inter}_2, n'_1, n'_2$ be # of cars at each location of each state. Since the two locations are independent, $p^{inter}_1$ and $p^{inter}_2$ can be computed independently.

For a location $i \in \{1, 2\}$, we define the probability to start the day with $n^{inter}$ and end it with $n'$ cars is :
$$P^{inter}(n^{inter}, n') = 
    \begin{cases}
        \displaystyle\sum_{k=(n^{inter} - n')^+}^{n^{inter}} p(k, \lambda^{rent}) p(k + n' - n^{inter}, \lambda^{return}) + \underbrace{\sum_{k \geq n^{inter} + 1} p(k, \lambda^{rent})}_{\substack{\text{probability of rent demand} \\ \text{exceeding the available number of cars } n^{inter}}} p(n', \lambda^{return}) & \text{if } n' < n_{max} \\
        \displaystyle\sum_{k=(n^{inter} - n')^+}^{n^{inter}} p(k, \lambda^{rent}) \underbrace{\sum_{l \geq k + n' - n^{inter}} p(l, \lambda^{return})}_{\substack{\text{probability of retuned cars} \\ \text{exceeding final available space (after rent)}}} + \sum_{k \geq n^{inter} + 1} p(k, \lambda^{rent}) \sum_{l \geq n'} p(l, \lambda^{return}) & \text{if } n' = n_{max}
    \end{cases} $$
where $p(k, \lambda) = \frac{\lambda^k}{k!} e^{-\lambda}$ is the poisson mass function of demand of rent or number of returns at a given location at rate $\lambda$.


Computing $p^{inter}_1$ and $p^{inter}_2$ we can deduce the probability of going from an intermediary state $s^{inter} := (n^{inter}_1, n'_1)$ to a final state $s' := (n'_1, n'_2)$ as $$P^{inter}(s^{inter}, s') = P^{inter}_1(n^{inter}_1, n'_1) \times P^{inter}_2(n^{inter}_2, n'_2)$$

Similarly, we compute rent revenue according to the expected number of rents at each locations starting from $s^{inter}$ and arriving at any possible state $n'$. Since any number of car rent demand has a positive probability, and since it is independent from number of returned cars, expected rent revenue is simply computed as:
$$ r^{rent}(n^{inter}) = c^{rent} \left ( \displaystyle\sum_{k=0}^{n^{inter}}kp(k, \lambda^{rent}) + n^{inter} \displaystyle\sum_{k\geq n^{inter}+1} p(k, \lambda^{rent}) \right )$$
So that the expected rent revenue over all locations is simply $R^{rent}(s^{inter}) = r^{rent}_1(n^{inter}_1) + r^{rent}_2(n^{inter}_2)$

Next step is to define legal moves for each state $s$ in order to compute the total expected reward function. Let $s := (n_1, n_2)$. Let $a$ be the number of cars moved from location 1 to location 2. Taking into account $n_{max}$ (the maximum number of cars that can be parked at a location) and $a_{max}$ (the maximum number of cars that can be moved from one location to the other), we have that :
$$ a \in \llbracket -\min(n_2, a_{max}, n_{max} - n_1), \min(n_1, a_{max}, n_{max} - n_2) \rrbracket$$

So that the total expected reward function can be computed as :
$$R(s, a) = R^{rent}(n_1-a, n_2+a) - c^{moves} |a|$$
In the modified versions of the problem, one might want to take into account the free ride that the employee proposes, or the cost of using a second parking lot when limitied space if available. These kind of nonlinearities are easily incorporated to the expected reward function with :
$$r^{free \ ride}(s, a) = 
    \begin{cases}
        c^{moves} & \text{if } a \geq 1 \\
        0 & \text{otherwise}
    \end{cases}$$

$$r^{parking}_i(n_i, a) = 
    \begin{cases}
        -c^{parking} & \text{if } n_i \pm a > n^{parking}_{max} \\
        0 &  \text{otherwise}
    \end{cases}
$$

Finally, the dynamics $P$ of the MDP can be computed thanks to $P^{inter}$ where :
$$P(s, a, s') = \displaystyle\sum_{s' \in S} P^{inter}(s + a, s')$$
where $s + a = (n_1 - a, n_2 + a)$

In [ ]:
# Build JCR problem
def buildJCRModel(maxCars, maxMoves, movingFee, lambdaRentLoc1, lambdaRentLoc2, lambdaRetLoc1, lambdaRetLoc2, freeRide, maxParkingSpace, parkingFee):
    nStates = (maxCars + 1) ** 2
    actionMapping = [np.arange(-min(min(nCarsLoc2, maxMoves), maxCars - nCarsLoc1) + maxMoves, min(min(nCarsLoc1, maxMoves), maxCars - nCarsLoc2) + 1 + maxMoves) for nCarsLoc1 in range(maxCars + 1) for nCarsLoc2 in range(maxCars + 1)]

    # PInterLoc1, PInterLoc2 : probability of going from nCarsInter to nCarsFinal after all moving of cars at Loc1 and Loc2 respectively
    PInterLoc1 = np.zeros((maxCars + 1, maxCars + 1))
    PInterLoc2 = np.zeros((maxCars + 1, maxCars + 1))
    for nCarsInter in range(maxCars + 1):
        for nCarsFinal in range(maxCars + 1):
            delta = nCarsFinal - nCarsInter
            if nCarsFinal < maxCars:
                for nCarsRented in range(max(0, -delta), nCarsInter + 1):
                    PInterLoc1[nCarsInter, nCarsFinal] += poisson.pmf(nCarsRented, lambdaRentLoc1) * poisson.pmf(nCarsRented + delta, lambdaRetLoc1)
                    PInterLoc2[nCarsInter, nCarsFinal] += poisson.pmf(nCarsRented, lambdaRentLoc2) * poisson.pmf(nCarsRented + delta, lambdaRetLoc2)
                PInterLoc1[nCarsInter, nCarsFinal] += poisson.pmf(nCarsFinal, lambdaRetLoc1) * (1 - poisson.cdf(nCarsInter, lambdaRentLoc1))
                PInterLoc2[nCarsInter, nCarsFinal] += poisson.pmf(nCarsFinal, lambdaRetLoc2) * (1 - poisson.cdf(nCarsInter, lambdaRentLoc2))

            else:
                for nCarsRented in range(nCarsInter + 1):
                    PInterLoc1[nCarsInter, maxCars] += poisson.pmf(nCarsRented, lambdaRentLoc1) * (1 - poisson.cdf(nCarsRented + delta - 1, lambdaRetLoc1))
                    PInterLoc2[nCarsInter, maxCars] += poisson.pmf(nCarsRented, lambdaRentLoc2) * (1 - poisson.cdf(nCarsRented + delta - 1, lambdaRetLoc2))
                PInterLoc1[nCarsInter, maxCars] += (1 - poisson.cdf(nCarsInter, lambdaRentLoc1)) * (1 - poisson.cdf(nCarsFinal - 1, lambdaRetLoc1))
                PInterLoc2[nCarsInter, maxCars] += (1 - poisson.cdf(nCarsInter, lambdaRentLoc2)) * (1 - poisson.cdf(nCarsFinal - 1, lambdaRetLoc2))

    # PInter
    PInter = np.zeros((nStates, nStates))
    for s1 in range(nStates):     
        for s2 in range(nStates):         
            nCarsInterLoc1, nCarsInterLoc2 = s1 // (maxCars + 1), s1 % (maxCars + 1)         
            nCarsFinalLoc1, nCarsFinalLoc2 = s2 // (maxCars + 1), s2 % (maxCars + 1)         
            PInter[s1, s2] = PInterLoc1[nCarsInterLoc1, nCarsFinalLoc1] * PInterLoc2[nCarsInterLoc2, nCarsFinalLoc2]

    # Transition : transition from state s to state sInter after moving a cars from Loc1 to Loc2
    nActionMax = 2 * maxMoves + 1
    transition = -np.ones((nStates, nActionMax), dtype=int)
    for s in range(nStates):     
        nCarsInterLoc1, nCarsInterLoc2 = s // (maxCars + 1), s % (maxCars + 1)     
        for a in actionMapping[s]:         
            transition[s, a] = nCarsInterLoc2 + a - maxMoves + (maxCars + 1) * (nCarsInterLoc1 - a + maxMoves)

    # RInter : expected rental revenue when starting the day at s (intermediary state after all moving of cars)
    RInter = np.zeros(nStates)
    for s in range(nStates):
        nCarsLoc1, nCarsLoc2 = s // (maxCars + 1), s % (maxCars + 1)
        rewardLoc1 = sum([10 * nCarsRented * poisson.pmf(nCarsRented, lambdaRentLoc1) for nCarsRented in range(nCarsLoc1 + 1)]) + 10 * nCarsLoc1 * (1 - poisson.cdf(nCarsLoc1, lambdaRentLoc1))
        rewardLoc2 = sum([10 * nCarsRented * poisson.pmf(nCarsRented, lambdaRentLoc2) for nCarsRented in range(nCarsLoc2 + 1)]) + 10 * nCarsLoc2 * (1 - poisson.cdf(nCarsLoc2, lambdaRentLoc2))
        RInter[s] = rewardLoc1 + rewardLoc2

    # Rewards : moving cost (and extra parking costs) + expected rental revenue
    rewardMapping = np.nan * np.ones((nStates, nActionMax))
    for s in range(nStates):
        for a in actionMapping[s]:
            sInter = transition[s, a]
            nCarsInterLoc1, nCarsInterLoc2 = sInter // (maxCars + 1), sInter % (maxCars + 1)
            rewardMapping[s, a] = -movingFee * abs(a - maxMoves) + freeRide * movingFee * (a - maxMoves >= 1) + RInter[sInter] + parkingFee * ((nCarsInterLoc1 > maxParkingSpace) + (nCarsInterLoc2 > maxParkingSpace))

    # Dynamics : transition probabilities of the MDP
    dynamics = np.zeros((nStates, nActionMax, nStates))
    for s in range(nStates):
        for a in actionMapping[s]:
            for sPrime in range(nStates):
                dynamics[s, a, sPrime] = PInter[transition[s, a], sPrime]

    # Terminal states
    terminals = []

    return nStates, actionMapping, rewardMapping, dynamics, terminals


In [ ]:
def plotPolicyJCR(policy):
    x = np.arange(maxCars + 1)
    y = np.arange(maxCars + 1)

    X, Y = np.meshgrid(x, y)
    Z = policy[X + (maxCars + 1) * Y] - maxMoves

    fig = go.Figure()
    fig.add_trace(go.Contour(x=x, y=x, z=Z, contours=dict(coloring="none", showlabels=True, start=-maxMoves, end=maxMoves, size=1), line=dict(color="black")))
    fig.update_layout(width=400, height=400)
    fig.show()

In [ ]:
def plotValueJCR(V):
    x = np.arange(maxCars + 1)
    y = np.arange(maxCars + 1)

    X, Y = np.meshgrid(x, y)
    Z = V[X + (maxCars + 1) * Y]

    fig = go.Figure()
    fig.add_trace(go.Surface(x=x, y=y, z=Z))
    fig.update_layout(width=400, height=400)
    
    fig.show()

In [ ]:
# Model
modelJCR = buildJCRModel(maxCars, maxMoves, movingFee, lambdaRentLoc1, lambdaRentLoc2, lambdaRetLoc1, lambdaRetLoc2, freeRide, maxParkingSpace, parkingFee)
gammaJCR = 0.9

In [ ]:
epsilonJCR = 0.001
vStarJCR, policyStarJCR = policyIter(modelJCR, epsilonJCR, gammaJCR, maxMoves * np.ones(nStatesJCR, dtype=int), np.zeros(nStatesJCR))
plotPolicyJCR(policyStarJCR)
plotValueJCR(vStarJCR)

### Value iteration

In [ ]:
def valueIter(model, theta, gamma, vInit):
    # Initialization
    nStates, actionMapping, rewardMapping, dynamics, terminals = model
    V = vInit.copy()
    V[terminals] = np.zeros(len(terminals))
    iter = 0
    converged = False
    while not converged:
        print(f"iteration {iter}")
        policy = []
        delta = 0
        for s in range(nStates):
            v = V[s]
            # Greedy action improv
            maxi = None
            for a in actionMapping[s]:
                qValAction = Ld(model, s, a, gamma, V)
                if maxi is None or qValAction > maxi:
                    maxi = qValAction
                    argmax = [a]
                elif qValAction == maxi:
                    argmax.append(a)

            policy.append(argmax)
            V[s] = qValAction
            delta = max(delta, abs(v - V[s]))

        converged = delta < theta      

        iter += 1

    return V, policy

#### Example 4.3 : Gambler's Problem

In [ ]:
def plotPolicyGP(policies):
    fig = go.Figure()
    fig.add_trace(go.Scatter(y=[rd.choice(actions) for actions in policies]))    
    fig.show()

In [ ]:
def plotValueGP(V):
    fig = go.Figure()
    fig.add_trace(go.Scatter(y=V[:-2]))    
    fig.show()

In [ ]:
def buildGPModel(goal, ph):
    nStates = goal + 1
    actionMapping = [np.arange(min(s, goal - s) + 1) for s in range(nStates)]
    nActionMax = goal + 1
    
    dynamics = np.zeros((nStates, nActionMax, nStates))
    for s in range(nStates):
        if s == 0 or s == goal:
            dynamics[s, :, s] = np.ones(nActionMax)
        if 0 < s < goal:
            for a in actionMapping[s]:
                dynamics[s, a, s + a] = ph
                dynamics[s, a, s - a] = 1 - ph

    rewardMapping = np.zeros((nStates, nActionMax))
    for s in range(nStates):
        if s < goal:
            for a in actionMapping[s]:
                rewardMapping[s, a] = dynamics[s, a, goal]

    terminals = [0, goal]

    return nStates, actionMapping, rewardMapping, dynamics, terminals

In [ ]:
goal = 100
ph = 0.99
gammaGP = 1
modelGP = buildGPModel(goal, ph)
vInitGP = np.zeros(goal + 1)
epsilonGP = 0.001
vStarGP, policyStarGP = valueIter(modelGP, epsilonGP, gammaGP, vInitGP)

In [ ]:
plotPolicyGP(policyStarGP)
plotValueGP(vStarGP)

## Monte Carlo Methods

### First-visit MC prediction for estimating $V \approx v_{\pi}$

In [ ]:
def plotLearningCurve(states, vStar, runStateValues):
    fig = go.Figure()
    for name, methodStateValues in runStateValues.items():
        for state, stateName in states.items():
            y = np.mean((np.stack([stateValues[:, state] for stateValues in methodStateValues]) - vStar[state])**2, axis=0)
            fig.add_trace(go.Scatter(x=np.arange(len(y)), y=y, name=f"{name} for state {stateName}"))
    fig.update_xaxes(type="log")
    fig.show()

In [ ]:
def firstVisitMCP(model, randomWalkFunc, gamma, maxIter, vInit, policy, ES=False, states=None, learningCurve=None):
    nStates, nActions, legalActions, actionMapping = model
    if states is not None:
        statesSet = set(states)
        coding = {state: i for i, state in enumerate(states)}
        V = vInit[states].copy()
        nVisits = np.zeros(len(states))
    else:
        V = vInit.copy()
        nVisits = np.zeros(nStates)

    if learningCurve is not None:
        learningCurve[0] = V.copy()

    iter = 0        
    while iter < maxIter:
        if iter % (maxIter // 5) == 0:
            print(f"Iteration {iter}")

        if ES:
            if states is not None:
                startingState = np.random.choice(states)
                startingAction = policy[coding[startingState]]
            else:
                startingState = np.random.randint(0, nStates)
                startingAction = policy[startingState]
            trajectory = randomWalkFunc(policy, startingState=startingState, startingAction=startingAction)
        else:
            trajectory = randomWalkFunc(policy)

        gain = 0.0
        T = len(trajectory)
        gains = np.zeros(T)
        for t in range(T - 1, -1, -1):
            gain = gamma * gain + trajectory[t, 2]
            gains[t] = gain

        visited = set()
        for t in range(T):
            currentState = trajectory[t, 0]
            if currentState in visited:
                continue
            visited.add(currentState)

            if states is not None:
                if currentState not in statesSet:
                    continue
                stateIndex = coding[currentState]

            else:
                stateIndex = currentState
                
            nVisits[stateIndex] += 1
            V[stateIndex] = V[stateIndex] + (gains[t] - V[stateIndex]) / nVisits[stateIndex]

        iter += 1
        if learningCurve is not None and iter < maxIter:
            learningCurve[iter] = V.copy()

    return V

#### Example 5.1 : BlackJack

In [ ]:
def codeBJState(hand, faceUp, usableAce):
    return 20 * (hand - 12) + 2 * (faceUp - 1) + usableAce

In [ ]:
def decodeBJState(state):    
    handNorm, r = divmod(state, 20)
    faceUpNorm, usableAce = divmod(r, 2)
    hand = handNorm + 12
    dealerCard = faceUpNorm + 1
    return hand, dealerCard, usableAce

In [ ]:
def sampleBJTrajectory(policy, startingState=None, startingAction=None):
    cards = np.arange(1, 11)
    cardsProb = [1 / 13] * 9 + [4 / 13]
    randomizedPolicy = len(policy.shape) == 2

    # Preping up the hands
    if startingState is None:
        playerFirstCard = np.random.choice(cards, p=cardsProb)
        playerSecondCard = np.random.choice(cards, p=cardsProb)
        # playerCards = [playerFirstCard, playerSecondCard]

        playerHand = playerFirstCard + playerSecondCard
        playerUsableAce = 1 * ((1 in [playerFirstCard, playerSecondCard]) and (playerHand + 10 <= 21))

        dealerFaceUpCard = np.random.choice(cards, p=cardsProb)

        while 10 * playerUsableAce + playerHand <= 11:
            newCard = np.random.choice(cards, p=cardsProb)
            playerHand += newCard
            # playerCards += [newCard]
            playerUsableAce = 1 * ((playerUsableAce or newCard == 1) and (10 + playerHand <= 21))

    else:
        playerHand, dealerFaceUpCard, playerUsableAce = decodeBJState(startingState)
        playerHand -= 10 * playerUsableAce
        # playerCards = [playerHand - 11 * playerUsableAce] + ([1] if playerUsableAce else [])


    dealerFaceDownCard = np.random.choice(cards, p=cardsProb)
    dealerHand = dealerFaceUpCard + dealerFaceDownCard
    dealerUsableAce = 1 * ((1 in [dealerFaceUpCard, dealerFaceDownCard]) and (dealerHand + 10 <= 21))
    # dealerCards = [dealerFaceUpCard, dealerFaceDownCard]

    stateTrajectory = []
    actionTrajectory = []

    # Game play
    if startingAction == 0: # hit, update hand
        stateTrajectory += [startingState]
        actionTrajectory += [0]
        # print(f"Player hand : {playerHand}, usable ace : {playerUsableAce}, total : {playerUsableAce * 10 + playerHand} --> player hits")
        newCard = np.random.choice(cards, p=cardsProb)
        playerHand += newCard
        # playerCards += [newCard]
        playerUsableAce = 1 * ((playerUsableAce or newCard == 1) and (10 + playerHand <= 21))
        # print(f"New card : {newCard}")

    if startingAction != 1:
        if 10 * playerUsableAce + playerHand <= 21:
            if randomizedPolicy:
                nextAction = np.random.choice([0, 1], p=policy[codeBJState(10 * playerUsableAce + playerHand, dealerFaceUpCard, playerUsableAce), :])
            else:
                nextAction = policy[codeBJState(10 * playerUsableAce + playerHand, dealerFaceUpCard, playerUsableAce)]

        while 10 * playerUsableAce + playerHand <= 21 and nextAction == 0:
            # print(f"Player hand : {playerHand}, usable ace : {playerUsableAce}, total : {playerUsableAce * 10 + playerHand} --> player hits")
            stateTrajectory += [codeBJState(10 * playerUsableAce + playerHand, dealerFaceUpCard, playerUsableAce)]
            actionTrajectory += [0]
            newCard = np.random.choice(cards, p=cardsProb)
            playerHand += newCard
            # playerCards += [newCard]
            playerUsableAce = 1 * ((playerUsableAce or newCard == 1) and (10 + playerHand <= 21))
            # print(f"New card : {newCard}")
            if 10 * playerUsableAce + playerHand <= 21:
                if randomizedPolicy:
                    nextAction = np.random.choice([0, 1], p=policy[codeBJState(10 * playerUsableAce + playerHand, dealerFaceUpCard, playerUsableAce), :])
                else:
                    nextAction = policy[codeBJState(10 * playerUsableAce + playerHand, dealerFaceUpCard, playerUsableAce)]
        
    if 10 * playerUsableAce + playerHand > 21:
        reward = -1

    else:
        stateTrajectory += [codeBJState(10 * playerUsableAce + playerHand, dealerFaceUpCard, playerUsableAce)]
        actionTrajectory += [1]

        while 10 * dealerUsableAce + dealerHand <= 16:
            newCard = np.random.choice(cards, p=cardsProb)
            dealerHand += newCard
            # dealerCards += [newCard]
            dealerUsableAce = 1 * ((dealerUsableAce or newCard == 1) and (10 + dealerHand <= 21))

        if 10 * dealerUsableAce + dealerHand > 21:
            reward = 1

        else:
            reward = - 1 * (10 * dealerUsableAce + dealerHand > 10 * playerUsableAce + playerHand) + 1 * (10 * dealerUsableAce + dealerHand < 10 * playerUsableAce + playerHand)

    trajectory = np.empty((len(stateTrajectory), 3), dtype=int)
    trajectory[:, 0] = stateTrajectory
    trajectory[:, 1] = actionTrajectory
    trajectory[:, 2] = [0] * (len(stateTrajectory) - 1) + [reward]

    return trajectory#, playerCards, dealerCards


In [ ]:
def plotPolicyBJ(policy):
    x = np.arange(1, 11)
    y = np.arange(12, 22)

    fig = make_subplots(rows=1, cols=2,
    subplot_titles=["no usable ace", "usable ace"])

    Xy1, Yy1 = np.meshgrid(x, y[1:])
    Xy_1, Yy_1 = np.meshgrid(x, y[:-1])
    Xx1, Yx1 = np.meshgrid(x[1:], y)
    Xx_1, Yx_1 = np.meshgrid(x[:-1], y)
    X, Y = np.meshgrid(x, y)

    for z in [0, 1]:
        if len(policy.shape) == 2:
            Z = policy[codeBJState(Y, X, z), 1]
            fig.add_trace(go.Heatmap(x=x, y=y, z=Z, colorbar=dict(x=0.45 if z ==0 else 1.02)), row=1, col=z+1)

        else:
            diff_x = policy[codeBJState(Yx1, Xx1, z)] != policy[codeBJState(Yx_1, Xx_1, z)]
            diff_y = policy[codeBJState(Yy1, Xy1, z)] != policy[codeBJState(Yy_1, Xy_1, z)]

            ix, iy = np.where(diff_x)
            for i, j in zip(ix, iy):
                fig.add_trace(
                    go.Scatter(
                        x=[1+j, 1+j],
                        y=[12+i-1, 12+i],
                        mode="lines",
                        line=dict(color="black"),
                        showlegend=False,
                        hoverinfo="none"
                    ),
                    row=1,
                    col=z+1,
                )

            ix, iy = np.where(diff_y)
            for i, j in zip(ix, iy):
                fig.add_trace(
                    go.Scatter(
                        x=[1+j-1, 1+j],
                        y=[12+i, 12+i],
                        mode="lines",
                        line=dict(color="black"),
                        showlegend=False,
                        hoverinfo="none"
                    ),
                    row=1,
                    col=z+1,
                )

            fig.update_xaxes(range=[0, 10], tickmode="array", tickvals=np.arange(11), ticks="inside", dtick=1, showticklabels=False, row=1, col=z+1)

            for i in range(11):
                fig.add_annotation(
                    x=i + 0.5,
                    y=0,
                    xref=f"x{2 if z == 1 else ''}",
                    yref=f"y{2 if z == 1 else ''} domain",
                    text=str(i + 1) if i >= 1 else "A" ,
                    showarrow=False,
                    yshift=-25,
                )

            fig.update_yaxes(range=[10, 21], tickvals=np.arange(10, 22), ticktext=np.arange(10, 22), ticks="inside", dtick=1, showticklabels=False, row=1, col=z+1)
                
            for i in range(10, 22):
                fig.add_annotation(
                    y=i + 0.5,
                    x=0,
                    xref=f"x{2 if z == 1 else ''}",
                    yref=f"y{2 if z == 1 else ''}",
                    text=str(i + 1),
                    showarrow=False,
                    xshift=-25,
                    row=1,
                    col=z+1,
                )
                
    fig.update_layout(showlegend=False)
    fig.show()

In [ ]:
def plotValueBJ(V):
    x = np.arange(12, 22)
    y = np.arange(1, 11)

    X, Y = np.meshgrid(x, y)
    Zusable = V[codeBJState(X, Y, 1)]
    ZnoUsable = V[codeBJState(X, Y, 0)]

    fig = make_subplots(rows=1, cols=2, specs=[[{'type': 'surface'}, {'type': 'surface'}]], subplot_titles=["no usable ace", "usable ace"])
    for z in [0, 1]:
        fig.add_trace(go.Surface(x=x, y=y, z=V[codeBJState(X, Y, z)], colorbar=dict(x=0.45 if z ==0 else 1.02)), row=1, col=z+1)

    fig.update_layout(showlegend=False)
    fig.show()

In [ ]:
# BlackJack model
nStatesBJ = 200
nActionsBJ = 2
actionMappingBJ = np.array([[0, 1] for s in range(nStatesBJ)])
legalActionsBJ = np.zeros((nStatesBJ, nActionsBJ), dtype=bool)
for state, action in enumerate(actionMappingBJ):
    legalActionsBJ[state, action] = True
modelBJ = nStatesBJ, nActionsBJ, legalActionsBJ, actionMappingBJ

In [ ]:
basePolicyBJ = np.empty(nStatesBJ, dtype=int)
for dealerFaceUpCard in range(1, 11):
    for usableAce in range(2):
        for playerHand in range(12, 20):
            basePolicyBJ[codeBJState(playerHand, dealerFaceUpCard, usableAce)] = 0
        basePolicyBJ[codeBJState(20, dealerFaceUpCard, usableAce)] = 1
        basePolicyBJ[codeBJState(21, dealerFaceUpCard, usableAce)] = 1

In [ ]:
plotPolicyBJ(basePolicyBJ)

In [ ]:
def decodeTrajectoryBJ(trajectory):
    decodedStates = []
    for s in trajectory[:, 0]:
        decodedStates += [decodeBJState(s)]
        
    decodedActions = []
    for a in trajectory[:, 1]:
        decodedActions += ["stick" * a + "hit" * (1 - a)]

    decoded = np.empty(trajectory.shape, dtype=object)
    decoded[:, 0] = decodedStates
    decoded[:, 1] = decodedActions
    decoded[:, 2] = trajectory[:, 2]

    return decoded

In [ ]:
nItersBJ = 500000
gammaBJ = 1
vInitBJ = np.zeros(nStatesBJ)
vBasePolicyBJ = firstVisitMCP(modelBJ, sampleBJTrajectory, gammaBJ, 1000000, vInitBJ, basePolicyBJ)

In [ ]:
plotValueBJ(vBasePolicyBJ)

In [ ]:
testStateBJ = codeBJState(13, 2, 1)
learningCurveMCPtestStateBJ = np.zeros((nItersBJ, 1))
vBasePolicyBJtestState = firstVisitMCP(modelBJ, sampleBJTrajectory, gammaBJ, nItersBJ, vInitBJ, basePolicyBJ, ES=True, states=[testStateBJ], learningCurve=learningCurveMCPtestStateBJ)

In [ ]:
plotLearningCurve({0: "(13, 2, usable ace)"}, vBasePolicyBJ[[testStateBJ]][:, None], {"First Visit MC Prediction": [learningCurveMCPtestStateBJ]})

### First-visit Monte Carlo ES (Exploring Starts), for estimating $\pi \approx \pi^*$

#### Example 5.3 : Solving BlackJack

In [ ]:
def firstVisitMCCES(model, randomWalkFunc, gamma, maxIter, qInit, policyInit):
    nStates, nActions, legalActions, actionMapping = model
    
    Q = qInit.copy()
    policy = policyInit.copy()
    nVisits = np.zeros((nStates, nActions))
    iter = 0

    while iter < maxIter:
        if iter % (maxIter // 5) == 0:
            print(f"Iteration {iter}")

        startingState = np.random.randint(0, nStates)
        startingAction = np.random.choice(actionMapping[startingState])

        trajectory = randomWalkFunc(policy, startingState=startingState, startingAction=startingAction)
        gain = 0.0
        T = len(trajectory)
        gains = np.zeros(T)
        for t in range(T - 1, -1, -1):
            gain = gamma * gain + trajectory[t, 2]
            gains[t] = gain
        visited = set()
        for t in range(T):
            currentState = trajectory[t, 0]
            currentAction = trajectory[t, 1]
            if (currentState, currentAction) in visited:
                continue
            visited.add((currentState, currentAction))

            nVisits[currentState, currentAction] += 1
            Q[currentState, currentAction] = Q[currentState, currentAction] + (gains[t] - Q[currentState, currentAction]) / nVisits[currentState, currentAction]
            policy[currentState] = np.argmax(Q[currentState, :])

        iter += 1

    return Q, policy

In [ ]:
nItersBJOpt = 1000000
qInitBJ = np.zeros((nStatesBJ, nActionsBJ))
policyInitBJ = np.ones(nStatesBJ, dtype=int)
qStarBJ, policyStarBJ = firstVisitMCCES(modelBJ, sampleBJTrajectory, gammaBJ, nItersBJOpt, qInitBJ, policyInitBJ)
vStarBJ = qStarBJ[np.arange(nStatesBJ), policyStarBJ]

In [ ]:
plotValueBJ(vStarBJ)
plotPolicyBJ(policyStarBJ)

### On-policy first-visit  Monte Carlo control (for $\epsilon$-soft policies), estimates $\pi \approx \pi^*$

In [ ]:
def generateSoftPolicy(states, legalActions, epsilon, Q):
    nLegalActions = legalActions.sum(axis=1)
    policy = legalActions.astype(float) / nLegalActions[:, None]
    legalStates = legalActions[states]
    maskedQ = np.where(legalStates, Q, -np.inf)
    aStar = np.argmax(maskedQ, axis=1)
    policy[states] = epsilon * legalStates / nLegalActions[states][:, None]
    policy[states, aStar] += 1 - epsilon
    return policy

In [ ]:
def onPolicyFVMCCEpsilonSoft(model, epsilon, randomWalkFunc, gamma, maxIter, qInit, policyInit):
    nStates, nActions, legalActions, actionMapping = model
    
    Q = qInit.copy()
    policy = policyInit.copy() # should be epsilon-soft ! 
    nVisits = np.zeros((nStates, nActions))
    lengths = legalActions.sum(axis=1)
    iter = 0

    while iter < maxIter:
        if iter % (maxIter // 5) == 0:
            print(f"Iteration {iter}")

        trajectory = randomWalkFunc(policy)
        gain = 0.0
        T = len(trajectory)
        gains = np.zeros(T)
        for t in range(T - 1, -1, -1):
            gain = gamma * gain + trajectory[t, 2]
            gains[t] = gain

        visited = set()
        for t in range(T):
            currentState = trajectory[t, 0]
            currentAction = trajectory[t, 1]
            if (currentState, currentAction) in visited:
                continue
            visited.add((currentState, currentAction))

            nVisits[currentState, currentAction] += 1
            Q[currentState, currentAction] = Q[currentState, currentAction] + (gains[t] - Q[currentState, currentAction]) / nVisits[currentState, currentAction]
            aStar = np.argmax(Q[currentState, :])
            policy[currentState] = epsilon * legalActions[currentState] / lengths[currentState]
            policy[currentState, aStar] += 1 - epsilon

        iter += 1
                    
    return Q, policy
    

#### $\epsilon$-soft policy for the BlackJack problem

In [ ]:
epsilonBJ = 0.001
epsilonSoftPolicyBJ = generateSoftPolicy(np.arange(nStatesBJ), legalActionsBJ, epsilonBJ, qInitBJ)
qEpsilonSoftStarBJ, policyEpsilonSoftStarBJ = onPolicyFVMCCEpsilonSoft(modelBJ, epsilonBJ, sampleBJTrajectory, gammaBJ, nItersBJOpt, qInitBJ, epsilonSoftPolicyBJ)

In [ ]:
vEpsilonSoftStarBJ = np.sum(policyEpsilonSoftStarBJ * qEpsilonSoftStarBJ, axis=1)
plotValueBJ(vEpsilonSoftStarBJ)
plotPolicyBJ(policyEpsilonSoftStarBJ)

### Off-policy MC prediction (policy evaluation) for estimating $Q \approx q_{\pi}$

In [ ]:
def transformToStochasticPi(nStates, nActions, policy):
    if len(policy.shape) == 1:
        stochasticPolicy = np.zeros((nStates, nActions), dtype=int)
        stochasticPolicy[np.arange(nStates), policy] = np.ones(nStates, dtype=int)
        return stochasticPolicy
    return policy

In [ ]:
def offPolicyEVMCP(model, randomWalkFunc, gamma, maxIter, target, qInit, epsilon=0.001, uniformBehaviour=False, weighted=True, ES=False, states=None, learningCurve=None, verbose=True):
    nStates, nActions, legalActions, actionMapping = model

    if states is not None:
        statesSet = set(states)
        coding = {state: i for i, state in enumerate(states)}
        Q = qInit[states, :].copy()
        nStatesFinal = len(states)
        statesFinal = states
    else:
        Q = qInit.copy()
        nStatesFinal = nStates
        statesFinal = np.arange(nStates)
    
    if weighted:
        ratios = np.zeros((nStatesFinal, nActions))
    else:
        nVisits = np.zeros((nStatesFinal, nActions))

    iter = 0
    targetPolicy = transformToStochasticPi(nStates, nActions, target)

    if uniformBehaviour:
        behaviourPolicy = legalActions.astype(float) / legalActions.sum(axis=1)[:, None]

    if learningCurve is not None:
        learningCurve[0] = np.sum(targetPolicy[statesFinal] * Q, axis=1)

    while iter < maxIter:
        if verbose and iter % (maxIter // 5) == 0:
            print(f"Iteration {iter}")

        if not uniformBehaviour:
            behaviourPolicy = generateSoftPolicy(statesFinal, legalActions, epsilon, Q)

        if ES:
            if states is not None:
                startingState = np.random.choice(states)
            else:
                startingState = np.random.randint(0, nStates)
            trajectory = randomWalkFunc(behaviourPolicy, startingState=startingState, startingAction=np.random.choice(actionMapping[startingState]))
        else:
            trajectory = randomWalkFunc(behaviourPolicy)

        gain = 0.0
        weight = 1
        t = len(trajectory) - 1
        while t >= 0:
            gain = gamma * gain + trajectory[t, 2]
            currentState = trajectory[t, 0]
            currentAction = trajectory[t, 1]

            if states is not None:
                if currentState not in statesSet:
                    t -= 1
                    continue
                stateIndex = coding[currentState]

            else:
                stateIndex = currentState

            if weighted:
                ratios[stateIndex, currentAction] += weight
                Q[stateIndex, currentAction] = Q[stateIndex, currentAction] + weight * (gain - Q[stateIndex, currentAction]) / ratios[stateIndex, currentAction]
            else:
                nVisits[stateIndex, currentAction] += 1
                Q[stateIndex, currentAction] = Q[stateIndex, currentAction] + (weight * gain - Q[stateIndex, currentAction]) / nVisits[stateIndex, currentAction]

            weight *= targetPolicy[currentState, currentAction] / behaviourPolicy[currentState, currentAction]
            t -= 1

        iter += 1
        if learningCurve is not None and iter < maxIter:
            learningCurve[iter] = np.sum(targetPolicy[statesFinal] * Q, axis=1)
                    
    return Q
    

In [ ]:
def offPolicyFVMCP(model, randomWalkFunc, gamma, maxIter, target, qInit, epsilon=0.001, uniformBehaviour=False, weighted=True, ES=False, states=None, learningCurve=None, verbose=True):
    nStates, nActions, legalActions, actionMapping = model

    if states is not None:
        statesSet = set(states)
        coding = {state: i for i, state in enumerate(states)}
        Q = qInit[states, :].copy()
        nStatesFinal = len(states)
        statesFinal = states
    else:
        Q = qInit.copy()
        nStatesFinal = nStates
        statesFinal = np.arange(nStates)
    
    if weighted:
        ratios = np.zeros((nStatesFinal, nActions))
    else:
        nVisits = np.zeros((nStatesFinal, nActions))

    iter = 0
    targetPolicy = transformToStochasticPi(nStates, nActions, target)

    if uniformBehaviour:
        behaviourPolicy = legalActions.astype(float) / legalActions.sum(axis=1)[:, None]

    if learningCurve is not None:
        learningCurve[0] = np.sum(targetPolicy[statesFinal] * Q, axis=1)

    while iter < maxIter:
        if verbose and iter % (maxIter // 5) == 0:
            print(f"Iteration {iter}")

        if not uniformBehaviour:
            behaviourPolicy = generateSoftPolicy(statesFinal, legalActions, epsilon, Q)

        if ES:
            if states is not None:
                startingState = np.random.choice(states)
            else:
                startingState = np.random.randint(0, nStates)
            trajectory = randomWalkFunc(behaviourPolicy, startingState=startingState, startingAction=np.random.choice(actionMapping[startingState]))
        else:
            trajectory = randomWalkFunc(behaviourPolicy)

        gain = 0.0
        T = len(trajectory)
        gains = np.zeros(T)
        weights = np.ones(T)
        for t in range(T - 1, -1, -1):
            gain = gamma * gain + trajectory[t, 2]
            gains[t] = gain
            if t != T - 1:
                weights[t] = weights[t + 1] * targetPolicy[trajectory[t + 1, 0], trajectory[t + 1, 1]] / behaviourPolicy[trajectory[t + 1, 0], trajectory[t + 1, 1]]

        visited = set()
        for t in range(T):
            currentState = trajectory[t, 0]
            currentAction = trajectory[t, 1]
            if (currentState, currentAction) in visited:
                continue
            visited.add((currentState, currentAction))

            if states is not None:
                if currentState not in statesSet:
                    continue
                stateIndex = coding[currentState]
            else:
                stateIndex = currentState

            if weighted:
                ratios[stateIndex, currentAction] += weights[t]
                Q[stateIndex, currentAction] = Q[stateIndex, currentAction] + weights[t] * (gains[t] - Q[stateIndex, currentAction]) / ratios[stateIndex, currentAction]
            else:
                nVisits[stateIndex, currentAction] += 1
                Q[stateIndex, currentAction] = Q[stateIndex, currentAction] + (weights[t] * gains[t] - Q[stateIndex, currentAction]) / nVisits[stateIndex, currentAction]

        iter += 1
        if learningCurve is not None and iter < maxIter:
            learningCurve[iter] = np.sum(targetPolicy[statesFinal] * Q, axis=1)

    return Q

#### Example 5.4 : Off-policy Estimation of a BlackJack State Value

In [ ]:
basePolicyTestStateBJ = transformToStochasticPi(200, 2, basePolicyBJ)
behaviourPolicyTestStateBJ = np.array([[0.5, 0.5]] * 200)
def OffPolicyFVBJ(nIters):
    Qweighted = np.zeros(2)
    Qordinary = np.zeros(2)
    ratios = np.zeros(2)
    nVisits = np.zeros(2)
    learningCurveWeighted = np.empty((nIters, 1))
    learningCurveOrdinary = np.empty((nIters, 1))

    iter = 0
    learningCurveWeighted[0, 0] = Qweighted[0]
    learningCurveOrdinary[0, 0] = Qordinary[0]

    while iter < nIters:
        firstAction = np.random.choice([0, 1])
        trajectory = sampleBJTrajectory(behaviourPolicyTestStateBJ, testStateBJ, firstAction)
        weight = 1.0
        T = len(trajectory) - 1
        gain = trajectory[T, 2]
        for t in range(T - 1 , -1, -1):
            weight *= basePolicyTestStateBJ[trajectory[t, 0], trajectory[t, 1]] / behaviourPolicyTestStateBJ[trajectory[t, 0], trajectory[t, 1]] 
            t -= 1
        
        nVisits[firstAction] += 1
        Qordinary[firstAction] += (weight * gain - Qordinary[firstAction]) / nVisits[firstAction]

        if weight > 0:
            ratios[firstAction] += weight
            Qweighted[firstAction] += weight * (gain - Qweighted[firstAction]) / ratios[firstAction]

        iter += 1
        if iter < nIters:
            learningCurveWeighted[iter, 0] = Qweighted[0]
            learningCurveOrdinary[iter, 0] = Qordinary[0]

    return learningCurveWeighted, learningCurveOrdinary

In [ ]:
def prepareExperimentsBJ(nRuns, nIters):
    weightedCurves = []
    ordinaryCurves = []
    for run in range(nRuns):
        print(f"Run {run}")
        runWeightedCurve, runOrdinaryCurve = OffPolicyFVBJ(nIters)
        weightedCurves.append(runWeightedCurve)
        ordinaryCurves.append(runOrdinaryCurve)
    return weightedCurves, ordinaryCurves

In [ ]:
WOPMCPtestStateBJ, OOPMCPtestStateBJ = prepareExperimentsBJ(100, 10000)

In [ ]:
plotLearningCurve({0: "(13, 2, usable ace)"}, vBasePolicyBJ[[testStateBJ]][:, None], {"Ordinary importance sampling": OOPMCPtestStateBJ,
                                                                                      "Weighted importance sampling": WOPMCPtestStateBJ})

#### Example 5.5 : Infinite Variance

In [ ]:
nStatesIV = 1
nActionsIV = 2
actionMappingIV = [[0, 1]] # 0 is left and 1 is right
legalActionsIV = np.array([[True, True]])
modelIV = nStatesIV, nActionsIV, legalActionsIV, actionMappingIV

In [ ]:
def sampleIVTrajectory(policy, startingState=None, startingAction=None):
    randomizedPolicy = len(policy.shape) == 2
    stateTrajectory = []
    actionTrajectory = []

    if startingAction == 1:
        stateTrajectory += [0]
        actionTrajectory += [1]
        reward = 0

    if startingAction == 0:
        stateTrajectory += [0]
        actionTrajectory += [0]
        nextState = np.random.choice([0, 1], p=[0.9, 0.1])
        if nextState == 1:
            reward = 1

    if startingAction is None or (startingAction == 0 and nextState == 0):
        if randomizedPolicy:
            nextAction = np.random.choice([0, 1], p=policy[0, :])
        else:
            nextAction = policy[0]

        while nextAction == 0:
            stateTrajectory += [0]
            actionTrajectory += [0]

            nextState = np.random.choice([0, 1], p=[0.9, 0.1])
            if nextState == 1:
                break

            if randomizedPolicy:
                nextAction = np.random.choice([0, 1], p=policy[0, :])
            else:
                nextAction = policy[0]

        if nextAction == 1:
            stateTrajectory += [0]
            actionTrajectory += [1]
            reward = 0

        else:
            reward = 1


    trajectory = np.empty((len(stateTrajectory), 3), dtype=int)
    trajectory[:, 0] = stateTrajectory
    trajectory[:, 1] = actionTrajectory
    trajectory[:, 2] = [0] * (len(stateTrajectory) - 1) + [reward]

    return trajectory


In [ ]:
gammaIV = 1
qInitIV = np.zeros((nStatesIV, nActionsIV))

In [ ]:
basePolicyIV = np.array([0])
behaviourPolicyIV = legalActionsIV.astype(float) / legalActionsIV.sum(axis=1)[:, None]

In [ ]:
OOPMCPtestStateIV = []
nItersIV = 1000000
for run in range(10):
    print(f"Run {run}")
    runOrdinaryCurve = np.empty((nItersIV, 1))
    offPolicyFVMCP(modelIV, sampleIVTrajectory, gammaIV, nItersIV, basePolicyIV, qInitIV, uniformBehaviour=True, weighted=False, learningCurve=runOrdinaryCurve, verbose=False)
    OOPMCPtestStateIV.append(runOrdinaryCurve)


In [ ]:
fig = go.Figure()
for run in range(10):
    fig.add_trace(go.Scatter(x=np.arange(nItersIV), y=OOPMCPtestStateIV[run][:, 0]))
fig.update_xaxes(type="log")
fig.update_layout(showlegend=False)
fig.show()

In [ ]:
OOPEVMCPtestStateIV = []
nItersIV = 1000000
for run in range(10):
    print(f"Run {run}")
    runOrdinaryCurve = np.empty((nItersIV, 1))
    offPolicyEVMCP(modelIV, sampleIVTrajectory, gammaIV, nItersIV, basePolicyIV, qInitIV, uniformBehaviour=True, weighted=False, learningCurve=runOrdinaryCurve, verbose=False)
    OOPEVMCPtestStateIV.append(runOrdinaryCurve)

In [ ]:
fig = go.Figure()
for run in range(10):
    fig.add_trace(go.Scatter(x=np.arange(nItersIV), y=OOPEVMCPtestStateIV[run][:, 0]))
fig.update_xaxes(type="log")
fig.update_layout(showlegend=False)
fig.show()

### Off-policy MC control, for estimating $\pi \approx \pi^*$

In [ ]:
def offPolicyEVMCC(model, randomWalkFunc, gamma, maxIter, policyInit, qInit, epsilon=0.001, uniformBehaviour=False):
    nStates, nActions, legalActions, actionMapping = model

    Q = qInit.copy()    
    ratios = np.zeros((nStates, nActions))

    iter = 0
    targetPolicy = policyInit.copy()

    if uniformBehaviour:
        behaviourPolicy = legalActions.astype(float) / legalActions.sum(axis=1)[:, None]

    while iter < maxIter:
        if iter % (maxIter // 5) == 0:
            print(f"Iteration {iter}")

        if not uniformBehaviour:
            behaviourPolicy = generateSoftPolicy(np.arange(nStates), legalActions, epsilon, Q)

        trajectory = randomWalkFunc(behaviourPolicy)

        gain = 0.0
        weight = 1
        t = len(trajectory) - 1
        while t >= 0:
            gain = gamma * gain + trajectory[t, 2]
            currentState = trajectory[t, 0]
            currentAction = trajectory[t, 1]

            ratios[currentState, currentAction] += weight
            Q[currentState, currentAction] = Q[currentState, currentAction] + weight * (gain - Q[currentState, currentAction]) / ratios[currentState, currentAction]

            targetPolicy[currentState] = np.argmax(Q[currentState, :], axis=1)
            if targetPolicy[currentState] != currentAction:
                t = -1
            else:
                weight *= 1 / behaviourPolicy[currentState, currentAction]
                t -= 1

        iter += 1
    
    return Q, targetPolicy

#### Exercice 5.12 : Racetrack

In [ ]:
def constructTrack(xMax, yMax):
    track = np.ones((yMax, xMax))
    starts = np.zeros(yMax)
    ends = xMax * np.ones(yMax)
    for y in range(yMax - 1, -1, -1):
        startMax = 2 * xMax // 3 if y == yMax - 1 else min(starts[y + 1] + 3, ends[y + 1])
        startMin = 0 if y == yMax - 1 else max(0, starts[y + 1] - 3)
        start = np.random.randint(startMin, startMax)
        endMinPrev = 1 if y == yMax - 1 else max(starts[y + 1] + 1, ends[y + 1])
        endMin = max(endMinPrev, start + 1)
        endMax = xMax if y == yMax - 1 else ends[y + 1] + 3
        end = np.random.randint(endMin, endMax + 1)
        track[y, :start] = 0
        track[y, end:] = 0
        starts[y] = start
        ends[y] = end

    track[-1, :] *= 2
    track[:, -1] *= 3
    return track

In [ ]:
def plotTrack(track):

    fig = go.Figure(
        go.Heatmap(
            z=track,
            colorscale = [
                [0.00, "#FFFFFF"],  # hors piste
                [0.33, "#FFFFFF"],

                [0.33, "#404040"],  # piste
                [0.66, "#404040"],

                [0.66, "#2196F3"],  # départ
                [0.99, "#2196F3"],

                [1.00, "#4CAF50"],  # arrivée
            ],
            showscale=False,
            xgap=1,
            ygap=1,
            hoverongaps=False,
        )
    )

    fig.update_layout(
        plot_bgcolor="white",
    )

    fig.update_yaxes(
        scaleanchor="x",
        autorange="reversed"
    )

    fig.show()


In [ ]:
track = constructTrack(30, 30)
plotTrack(track)